In [1]:
!pip install pandas duckdb jupysql

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.8/192.8 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.6/668.6 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.5/202.5 kB 12.2 MB/s eta 0:00:00


In [2]:
import pandas as pd
import sqlite3

base_raw_url = "https://raw.githubusercontent.com/isaakzyjiang/BAML-coursework/main/SQL/datas/"

files = ["addresses.csv", "customers.csv", "orders.csv", "products.csv", "suppliers.csv"]

conn = sqlite3.connect("mydatabase.db")

print("Starting data migration from GitHub...")

for file_name in files:
    url = base_raw_url + file_name
    table_name = file_name.replace(".csv", "")
    
    try:
        df = pd.read_csv(url)
        df.to_sql(table_name, conn, if_exists='replace', index=False)
        print(f"Successfully imported table: [{table_name}]")
    except Exception as e:
        print(f"Failed to import {file_name}: {e}")

# Close the connection after processing
conn.close()
print("\nAll data is now stored in 'mydatabase.db'. You can start writing SQL!")

Starting data migration from GitHub...
Successfully imported table: [addresses]
Successfully imported table: [customers]
Successfully imported table: [orders]
Successfully imported table: [products]
Successfully imported table: [suppliers]

All data is now stored in 'mydatabase.db'. You can start writing SQL!


In [ ]:
%load_ext sql

%sql sqlite:///mydatabase.db

%config SqlMagic.autopandas = True

Fontconfig warning: ignoring UTF-8: not a valid region tag


Connecting to 'sqlite:///mydatabase.db'

Check if it works: simple select * from products

In [7]:
%%sql

select *
from products
limit 5

Running query in 'sqlite:///mydatabase.db'

,ID,Name,Stock,SellingPrice,SupplierPrice,SupplierID
0,1,Iphone,20,1200,1000,1
1,14,Laptop,18,800,600,2
2,32,Headphones,3,600,400,3
3,64,TV,5,1400,1200,4
4,754,Smartwatch,18,301,62,2


Select all products with supplier cost greater 200 and sort them according to the costs in descending order.

In [9]:
%%sql

select *
from products
where SupplierPrice > 200
order by SupplierPrice DESC

Running query in 'sqlite:///mydatabase.db'

,ID,Name,Stock,SellingPrice,SupplierPrice,SupplierID
0,64,TV,5,1400,1200,4
1,1,Iphone,20,1200,1000,1
2,14,Laptop,18,800,600,2
3,396,VR Headset,7,596,487,2
4,703,Power Bank,1,736,464,5
5,877,Monitor,22,565,407,3
6,32,Headphones,3,600,400,3
7,171,Fitness Tracker,50,496,388,1
8,373,Smart Thermostat,11,573,361,2
9,467,Router,3,476,359,6


Select all orders from April, 2025 with an order total below 1000 euros. Sort according to order total ascending order

In [18]:
%%sql

select *
from orders
where Date >= '2025-04-01' and Date < '2025-05-01'
and (Qty*UnitPrice) < 1000
order by (Qty*UnitPrice) ASC

Running query in 'sqlite:///mydatabase.db'

,OrderID,Date,ProductID,CustomerID,ShippingAddressID,IBAN,Qty,UnitPrice
0,10951,2025-04-28,444,8770,2,GB76SORK24914168601347,1,244.00
1,10287,2025-04-27,646,4630,28,GB79MTUQ49027703843098,1,301.00
2,10409,2025-04-14,754,6119,13,GB93IQVN40155778817884,1,301.00
3,10562,2025-04-06,754,8560,38,GB93NFNI48869461195243,1,301.00
4,10708,2025-04-05,754,6119,24,GB77XHBC02724055934296,1,301.00
5,10995,2025-04-27,646,8381,3,GB18XXIF44626325144881,1,301.00
6,10979,2025-04-26,350,8560,8,GB42VCYK06662670551437,1,359.00
7,10658,2025-04-24,833,1710,3,GB64GIFG46199423767694,1,370.00
8,10843,2025-04-06,833,4886,6,GB70WWZP43811995376601,1,370.00
9,10883,2025-04-17,180,8438,4,GB44YTNB33582686270369,1,391.76


Calculate the total revenue in April 2025

In [20]:
%%sql

select sum(Qty*UnitPrice) as TotalRevenue
from orders
where Date >= '2025-04-01' and Date < '2025-05-01'

Running query in 'sqlite:///mydatabase.db'

,TotalRevenue
0,119253.71


Select the total revenue per customer

In [23]:
%%sql

select CustomerID, sum(Qty*UnitPrice) as TotalRevenue
from orders
group by CustomerID

Running query in 'sqlite:///mydatabase.db'

,CustomerID,TotalRevenue
0,1018,44425.99
1,1042,44674.28
2,1710,49184.01
3,1841,36577.16
4,2501,50356.33
5,2612,53557.18
6,3248,55952.35
7,3344,40017.08
8,3939,35599.39
9,4630,41163.65


Select the total revenue per customer but include only customers with an average revenue below 5000

In [24]:
%%sql

select CustomerID, sum(Qty*UnitPrice) as TotalRevenue
from orders
group by CustomerID
having avg(Qty*UnitPrice) < 5000

Running query in 'sqlite:///mydatabase.db'

,CustomerID,TotalRevenue
0,1018,44425.99
1,1042,44674.28
2,1710,49184.01
3,1841,36577.16
4,2501,50356.33
5,2612,53557.18
6,3248,55952.35
7,3344,40017.08
8,3939,35599.39
9,4630,41163.65


Calculate revenue per product. The output should include the product name and the total revenue per product. Sort the output by revenue, descending order.

In [31]:
%%sql

select P.Id, P.Name, sum(O.Qty*O.UnitPrice) as TotalRevenue
from products P, orders O
where P.ID = O.ProductID
group by P.ID
order by TotalRevenue DESC

Running query in 'sqlite:///mydatabase.db'

,ID,Name,TotalRevenue
0,1,Iphone,170130.25
1,64,TV,143109.06
2,703,Power Bank,114355.75
3,14,Laptop,108331.78
4,489,E-Reader,82766.92
5,877,Monitor,78463.78
6,467,Router,74393.75
7,373,Smart Thermostat,72601.74
8,733,Smart Light,64305.01
9,32,Headphones,62101.00


Get the number of products ordered per country.

In [36]:
%%sql

select C.country, sum(O.Qty) as ProductCount
from customers C, orders O
where C.CustomerID = O.CustomerID
group by C.country

Running query in 'sqlite:///mydatabase.db'

,country,ProductCount
0,France,810
1,Germany,799
2,Italy,470
3,Spain,685
4,United States,233


Find all orders with their shipping address details.

In [43]:
%%sql

select O.OrderID, A.*
from orders O, addresses A
where O.CustomerID = A.CustomerID
group by O.OrderID

Running query in 'sqlite:///mydatabase.db'

,OrderID,CustomerID,Street,City,State,PostalCode,Country,AddressID
0,10000,8438,"11, rue de Leleu",Chevalier,None,16475,France,4
1,10001,8560,"24, rue Océane Gimenez",Sainte Maurice,None,330,France,38
2,10002,6119,Canale Gionata 1 Piano 8,Costalonga del friuli,Campobasso,60256,Italy,24
3,10003,4886,Lilian-Mülichen-Straße 11,Jessen,Sachsen-Anhalt,70106,Germany,14
4,10004,1018,"701, avenue Vidal",Sainte Alex,None,37556,France,37
...,...,...,...,...,...,...,...,...
995,10995,8381,"689, boulevard Sylvie Fouquet",Launaydan,None,77434,France,33
996,10996,7594,Klappplatz 1,Perleberg,Mecklenburg-Vorpommern,76483,Germany,5
997,10997,2501,Heßplatz 3,Leipziger Land,Bremen,75107,Germany,19
998,10998,7594,Klappplatz 1,Perleberg,Mecklenburg-Vorpommern,76483,Germany,5


Calculate revenue per supplier. The output should include the supplier name and the total revenue.

In [49]:
%%sql

select S.Name, sum(O.Qty*O.UnitPrice) as TotalRevenue
from suppliers S, products P, orders O
where S.SupplierID = P.SupplierID and P.ID = O.ProductID
group by S.Name


Running query in 'sqlite:///mydatabase.db'

,Name,TotalRevenue
0,Abbott-Munoz,187082.56
1,Blake and Sons,116724.04
2,Davis and Sons,197122.67
3,Doyle Ltd,306648.69
4,"Gardner, Robinson and Lawrence",41310.00
5,"Guzman, Hoffman and Baldwin",214900.77
6,"Henderson, Ramirez and Lewis",40551.20
7,"Mcclain, Miller and Henderson",254844.27
8,"Rodriguez, Figueroa and Sanchez",192342.55


Select all customers who ordered at least one product with Qty > *3*

In [55]:
%%sql

select *
from customers C
where exists (
    select *
    from orders O
    where O.CustomerID = C.CustomerID
    and O.Qty > 3
)


Running query in 'sqlite:///mydatabase.db'

,CustomerID,CustomerName,country
0,3939,Agnes Trub,Germany
1,1841,Veronica Naser,Germany
2,8381,Elizabeth Chapman,United States
3,8438,Zacharie Rey,France
4,7594,Philippe-Richard Teixeira,France
5,8245,Éric-Stéphane Nguyen,France
6,1018,Henrike Bloch B.A.,Germany
7,8560,Moisés Correa Valentín,Spain
8,1042,Miroslawa Huhn,Germany
9,4912,Fátima Guillen-Reig,Spain


Join the data of all tables from May 2025 into a single table.

In [56]:
%%sql

select *
from orders O, customers C, addresses A, products P, suppliers S
where S.SupplierID = P.SupplierID
and P.ID = O.ProductID
and A.CustomerID = O.CustomerID
and C.CustomerID = O.CustomerID
and O.Date >= '2025-05-01' and O.Date < '2025-06-01'

Running query in 'sqlite:///mydatabase.db'

,OrderID,Date,ProductID,CustomerID,ShippingAddressID,IBAN,Qty,UnitPrice,CustomerID,CustomerName,...,AddressID,ID,Name,Stock,SellingPrice,SupplierPrice,SupplierID,SupplierID,Name,Email
0,10112,2025-05-01,733,7594,35,GB09QJCI38424981829922,2,444.0,7594,Philippe-Richard Teixeira,...,5,733,Smart Light,46,444,345,6,6,Abbott-Munoz,abbott-munoz@example.com
1,10112,2025-05-01,733,7594,35,GB09QJCI38424981829922,2,444.0,7594,Philippe-Richard Teixeira,...,35,733,Smart Light,46,444,345,6,6,Abbott-Munoz,abbott-munoz@example.com


List orders where the UnitPrice is greater than the SelingPrice.

In [60]:
%%sql

select *
from orders O
where UnitPrice > (
    select SellingPrice
    from products P
    where O.productID = P.ID
)

Running query in 'sqlite:///mydatabase.db'

,OrderID,Date,ProductID,CustomerID,ShippingAddressID,IBAN,Qty,UnitPrice
0,10096,2024-06-27,132,2612,10,GB92NCHO50606853615305,2,228.69
1,10139,2025-04-30,754,1018,7,GB70XYDB69033119079030,4,360.08
2,10174,2025-02-26,733,6119,15,GB58RLRY07836134074517,5,478.97
3,10198,2024-10-28,373,7566,2,GB28KNDR53747411249035,5,629.86
4,10299,2024-08-06,14,7594,35,GB43RSBM12034717545588,5,903.58
5,10312,2024-09-27,14,8486,5,GB93YOHX03718376452088,3,954.42
6,10319,2025-01-11,489,1042,39,GB41CBEF03173109678994,3,530.76
7,10439,2024-10-15,180,1710,17,GB93DMOK67700284276246,5,477.21
8,10513,2024-07-14,877,8381,3,GB69VAWV89487060566418,5,584.14
9,10521,2024-12-13,489,8245,36,GB31RYEE74954508920388,4,533.66
